# Lab | Web Scraping

Welcome to the "Books to Scrape" Web Scraping Adventure Lab!

**Objective**

In this lab, we will embark on a mission to unearth valuable insights from the data available on Books to Scrape, an online platform showcasing a wide variety of books. As data analyst, you have been tasked with scraping a specific subset of book data from Books to Scrape to assist publishing companies in understanding the landscape of highly-rated books across different genres. Your insights will help shape future book marketing strategies and publishing decisions.

**Background**

In a world where data has become the new currency, businesses are leveraging big data to make informed decisions that drive success and profitability. The publishing industry, much like others, utilizes data analytics to understand market trends, reader preferences, and the performance of books based on factors such as genre, author, and ratings. Books to Scrape serves as a rich source of such data, offering detailed information about a diverse range of books, making it an ideal platform for extracting insights to aid in informed decision-making within the literary world.

**Task**

Your task is to create a Python script using BeautifulSoup and pandas to scrape Books to Scrape book data, focusing on book ratings and genres. The script should be able to filter books with ratings above a certain threshold and in specific genres. Additionally, the script should structure the scraped data in a tabular format using pandas for further analysis.

**Expected Outcome**

A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`. The function should scrape book data from the "Books to Scrape" website and return a `pandas` DataFrame with the following columns:

**Expected Outcome**

- A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`.
- The function should return a DataFrame with the following columns:
  - **UPC**: The Universal Product Code (UPC) of the book.
  - **Title**: The title of the book.
  - **Price (£)**: The price of the book in pounds.
  - **Rating**: The rating of the book (1-5 stars).
  - **Genre**: The genre of the book.
  - **Availability**: Whether the book is in stock or not.
  - **Description**: A brief description or product description of the book (if available).
  
You will execute this script to scrape data for books with a minimum rating of `4.0 and above` and a maximum price of `£20`. 

Remember to experiment with different ratings and prices to ensure your code is versatile and can handle various searches effectively!

**Resources**

- [Beautiful Soup Documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Pandas Documentation](https://pandas.pydata.org/pandas-docs/stable/index.html)
- [Books to Scrape](https://books.toscrape.com/)


**Hint**

Your first mission is to familiarize yourself with the **Books to Scrape** website. Navigate to [Books to Scrape](http://books.toscrape.com/) and explore the available books to understand their layout and structure. 

Next, think about how you can set parameters for your data extraction:

- **Minimum Rating**: Focus on books with a rating of 4.0 and above.
- **Maximum Price**: Filter for books priced up to £20.

After reviewing the site, you can construct a plan for scraping relevant data. Pay attention to the details displayed for each book, including the title, price, rating, and availability. This will help you identify the correct HTML elements to target with your scraping script.

Make sure to build your scraping URL and logic based on the patterns you observe in the HTML structure of the book listings!


---

**Best of luck! Immerse yourself in the world of books, and may the data be with you!**

**Important Note**:

In the fast-changing online world, websites often update and change their structures. When you try this lab, the **Books to Scrape** website might differ from what you expect.

If you encounter issues due to these changes, like new rules or obstacles preventing data extraction, don’t worry! Get creative.

You can choose another website that interests you and is suitable for scraping data. Options like Wikipedia, The New York Times, or even library databases are great alternatives. The main goal remains the same: extract useful data and enhance your web scraping skills while exploring a source of information you enjoy. This is your opportunity to practice and adapt to different web environments!

In [24]:
# 
!pip install requests beautifulsoup4 pandas

In [25]:
import requests
import pandas as pd

from bs4 import BeautifulSoup

In [26]:
url = "https://books.toscrape.com/"

response = requests.get(url)

soup = BeautifulSoup(response.text, "html.parser")

print(soup.title.text)


    All products | Books to Scrape - Sandbox



In [27]:
# book scrapping

rating_mapping = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

In [28]:
rating_element = soup.select_one("p.star-rating")

rating_class = rating_element["class"][1]

rating = rating_mapping[rating_class]

print(rating)

3


In [29]:
# extracting information from one product page:

book_url = "https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html"

response = requests.get(book_url)

soup = BeautifulSoup(response.text, "html.parser")

title = soup.h1.text

price = soup.select_one(".price_color").text

print("Title:", title)
print("Price:", price)

Title: A Light in the Attic
Price: Â£51.77


In [30]:

import requests
import pandas as pd

from bs4 import BeautifulSoup
from urllib.parse import urljoin


def scrape_books(min_rating, max_price):

    base_url = "https://books.toscrape.com/"
    current_url = base_url

    rating_mapping = {
        "One": 1,
        "Two": 2,
        "Three": 3,
        "Four": 4,
        "Five": 5
    }

    books_data = []

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    while current_url:

        response = requests.get(
            current_url,
            headers=headers,
            timeout=20
        )

        response.raise_for_status()

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        books = soup.select("article.product_pod")

        for book in books:

            # Extract rating
            rating_class = book.select_one(
                "p.star-rating"
            )["class"][1]

            rating = rating_mapping[rating_class]

            # Extract price
            price_text = book.select_one(
                ".price_color"
            ).text

            price = float(
                price_text.replace("£", "").strip()
            )

            # Apply rating and price filters
            if rating < min_rating or price > max_price:
                continue

            # Extract title and product URL
            title = book.h3.a["title"]

            product_url = urljoin(
                current_url,
                book.h3.a["href"]
            )

            # Visit individual product page
            product_response = requests.get(
                product_url,
                headers=headers,
                timeout=20
            )

            product_response.raise_for_status()

            product_soup = BeautifulSoup(
                product_response.text,
                "html.parser"
            )

            # Extract UPC from product information table
            table = product_soup.select_one(
                "table.table-striped"
            )

            product_details = {}

            for row in table.select("tr"):
                key = row.th.text.strip()
                value = row.td.text.strip()

                product_details[key] = value

            upc = product_details.get("UPC")

            # Extract genre from breadcrumb
            breadcrumbs = product_soup.select(
                "ul.breadcrumb li a"
            )

            genre = (
                breadcrumbs[-1].text.strip()
                if breadcrumbs
                else "Unknown"
            )

            # Extract availability
            availability = product_soup.select_one(
                ".availability"
            ).text.strip()

            # Extract description
            description_element = product_soup.select_one(
                "#product_description + p"
            )

            description = (
                description_element.text.strip()
                if description_element
                else "No description available"
            )

            # Store book information
            books_data.append({
                "UPC": upc,
                "Title": title,
                "Price (£)": price,
                "Rating": rating,
                "Genre": genre,
                "Availability": availability,
                "Description": description
            })

        # Navigate to next catalogue page
        next_button = soup.select_one(
            "li.next a"
        )

        if next_button:
            current_url = urljoin(
                current_url,
                next_button["href"]
            )
        else:
            current_url = None

    # Convert results into DataFrame
    df = pd.DataFrame(books_data)

    return df


In [31]:
# understand utrl 
url = "https://books.toscrape.com/"

In [32]:
page_1 = "https://books.toscrape.com/catalogue/page-1.html"

page_2 = "https://books.toscrape.com/catalogue/page-2.html"

In [33]:
#
import requests
from bs4 import BeautifulSoup

url = "https://books.toscrape.com/"

response = requests.get(url)
response.raise_for_status()

soup = BeautifulSoup(response.content, "html.parser")

# Find all books on the first page
books = soup.select("article.product_pod")

print("Number of books:", len(books))

# Inspect the first book
first_book = books[0]

print("Title:", first_book.h3.a["title"])
print("Price:", first_book.select_one(".price_color").text)
print("Rating:", first_book.select_one(".star-rating")["class"])
print("Availability:", first_book.select_one(".availability").text.strip())


Number of books: 20
Title: A Light in the Attic
Price: £51.77
Rating: ['star-rating', 'Three']
Availability: In stock


In [34]:
import requests
from bs4 import BeautifulSoup

In [35]:
# connect to the website 
url = "https://books.toscrape.com/"

response = requests.get(url)

soup = BeautifulSoup(response.text, "html.parser")

In [36]:
# find all books 

books = soup.select("article.product_pod")

print(len(books))

20


In [37]:
# define the book
book = books[0]


In [38]:
# book price 
price_element = book.select_one(".price_color")

if price_element:
    price_text = price_element.get_text(strip=True)

    # Remove unwanted characters
    price_text = price_text.replace("Â", "")
    price_text = price_text.replace("£", "").strip()

    price = float(price_text)

else:
    price = None

print("Book price:", price)


Book price: 51.77


In [39]:
import re

price_element = book.select_one(".price_color")

if price_element:
    price_text = price_element.get_text(strip=True)

    match = re.search(r"\d+\.\d+", price_text)

    price = float(match.group()) if match else None

else:
    price = None

print("Book price:", price)

Book price: 51.77
